In [ ]:
import sys,os
from datetime import datetime
from pathlib import Path
import json
import torch # 张量、自动求导等
import torch.nn as nn # 网络层，Conv1D、LSTM等
import torch.nn.functional as F # 函数层， ReLU、cross_entropy等
from torchinfo import summary
from torch.utils.data import DataLoader,TensorDataset
import time
import gc
from sklearn.metrics import roc_auc_score
import re
import math
import mne
from train import learning_rate, hidden_size
import numpy as np

root = Path(r"C:\Users\hank\Desktop\Research Reproduction\dianxian\self")
chb_data_path = Path(r"D:\data\dianxian\channal_select\chb03")

if str(root) not in sys.path:
    sys.path.append(str(root))

import tools as t

log_dir = root / "train_log"
embeddings = root / 'embeddings'
weight = root / 'weight'
Fp1_F7 = embeddings / 'Fp1-F7'
Fp2_F8 = embeddings / 'Fp2-F8'
T7_P7 = embeddings / 'T7-P7'
T8_P8 = embeddings / 'T8-P8'
P7_O1 = embeddings / 'P7-O1'
P8_O2 = embeddings / 'P8-O2'

out_channals = 32
batch = 32
elec = 6
embed_dim = 32
num_heads = 2
window_length = 10 * 256
window_overlap = 5 * 256
elec_names = ['Fp1-F7', 'Fp2-F8', 'T7-P7', 'T8-P8', 'P7-O1', 'P8-O2']

In [ ]:
def get_positions(elec_names,montage_name="standard_1020"):
    montage = mne.channels.make_standard_montage(montage_name)
    ch_list, dig_list = montage.ch_names, montage.dig
    pos = []
    for name in elec_names:
        a, b = name.split("-")
        # standard_1020 里是 'Fp1'、'F7'，注意大小写
        va = np.array(dig_list[ch_list.index(a)]['r'])
        vb = np.array(dig_list[ch_list.index(b)]['r'])
        pos.append((va + vb) / 2.0)
    return torch.tensor(np.stack(pos), dtype=torch.float32) # (6,3)

def build_adjacency(elec_names):
    """距离倒数邻接矩阵，对角线为 0"""
    p = get_positions(elec_names)
    n = len(elec_names)
    adj = torch.zeros(n, n)
    for i in range(n):
        for j in range(n):
            if i != j:
                d = torch.norm(p[i] - p[j])
                if d > 0:
                    adj[i, j] = 1.0 / d
    return adj  # (6, 6)

In [ ]:
class GraphConstructor(nn.Module):
    def __init__(self, num_nodes=6, hidden_dim=16, pos_adj=build_adjacency(elec_names)):
        super().__init__()
        self.num_nodes = num_nodes
        # 全局静态图
        self.adj_weights = nn.Parameter(torch.eye(num_nodes))
        # 打分MLP
        self.similarity_net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ELU(inplace=False),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid())
        # 物理图权重
        self.pos_factor = nn.Parameter(torch.tensor(0.5))
        # 冻结物理图，不参与训练
        # 常量
        self.register_buffer("pos_encoding", pos_adj.clone())

    # 局部动态图
    def compute_similarity(self, x):          # x: (B, N, D)
        B, N, D = x.shape
        node_i = x.unsqueeze(2).expand(-1, -1, N, -1)   # (B, N, N, D)
        node_j = x.unsqueeze(1).expand(-1, N, -1, -1)
        pairs = torch.cat([node_i, node_j], dim=-1) # 2*D
        # 打分
        return self.similarity_net(pairs).squeeze(-1) # (B,N,N)

    # 实际上原始输入是B,N,L，还要去适配卷积，变成B,N,L1,C，然后这个C才是hiddensize，而且这个同时也是输入是吧，最终两个图卷积才是C->?->C

    # 也即是原来的hidden_size是我x的卷积输出的特征数，是定死32的，而且也是我第一层图卷积的输入
    def forward(self, x):                      # x: (B, N, D)
        static_adj = F.softmax(self.adj_weights, dim=1)
        dynamic_adj = self.compute_similarity(x)
        pos_adj = F.softmax(self.pos_encoding, dim=1)
        return F.softmax(static_adj + dynamic_adj + self.pos_factor * pos_adj, dim=2)

m = GraphConstructor(num_nodes=6,hidden_dim=16,pos_adj=build_adjacency(elec_names))
summary(m,input_size=(32,6,16))

In [ ]:
class GraphConvolution(nn.Module):
    def __init__(self,in_channals,out_channals,hidden_size=16):
        super().__init__()
        # 图
        self.gc = GraphConstructor(hidden_size=hidden_size)
        # 权重
        self.fc = nn.Linear(in_channals,out_channals)

    def forward(self,x):
        # 图构建
        adjacency = self.gc(x)
        # 聚合
        x = torch.bmm(adjacency,x)
        return self.fc(x)


In [ ]:
from models import CommonConv1d,BILSTMClassify

In [ ]:
class Detection(nn.Module):
    def __init__(self,embed_dim=16,num_heads=2):
        super().__init__()
        self.gcn1 = GraphConvolution(hidden_size=1)
        self.gcn2 = GraphConvolution(hidden_size=1)
        self.common_conv = CommonConv2d()
        self.Bilstm = BILSTMClassify()
        self.atten = nn.MultiheadAttention(embed_dim, num_heads=num_heads, batch_first=True)

    def forward(self,x): # (B,6,L)
        x1 = x.unsqueeze(1) # (B,1,6,L)
        cc = self.common_conv(x1) # (B,C,6,L)


In [ ]:
class GCNBranch(nn.Module):

    def __init__(
        self,
        elec_names,
        input_dim=32,
        hidden_dim=32,
        time_steps=80
    ):
        super().__init__()

        self.time_steps = time_steps
        self.input_dim = input_dim

        pos_adj = build_adjacency(elec_names)

        # 每个时间片的原始 EEG 局部片段映射为节点特征
        self.node_projection = nn.Linear(input_dim, hidden_dim)

        self.gcn1 = GraphConvolution(
            hidden_dim,
            hidden_dim,
            num_nodes=len(elec_names),
            pos_adj=pos_adj
        )

        self.gcn2 = GraphConvolution(
            hidden_dim,
            hidden_dim,
            num_nodes=len(elec_names),
            pos_adj=pos_adj
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        # x: (B, 6, 2560)
        B, N, L = x.shape

        if L % self.time_steps != 0:
            raise ValueError(
                f"输入长度 {L} 必须能被 time_steps={self.time_steps} 整除"
            )

        # 每32个采样点作为一个局部时间片
        # (B, 6, 80, 32)
        x = x.reshape(
            B, N, self.time_steps, L // self.time_steps
        )

        # (B, 80, 6, 32)
        x = x.permute(0, 2, 1, 3).contiguous()

        # (B, 80, 6, 32) -> (B*80, 6, 32)
        x = x.reshape(
            B * self.time_steps,
            N,
            self.input_dim
        )

        x = self.node_projection(x)

        # 第一层图卷积
        residual = x
        x = self.gcn1(x)
        x = F.relu(self.norm1(x + residual))

        # 第二层图卷积
        residual = x
        x = self.gcn2(x)
        x = F.relu(self.norm2(x + residual))

        # (B*80, 6, 32) -> (B, 80, 6, 32)
        x = x.reshape(
            B,
            self.time_steps,
            N,
            -1
        )

        return x


# ============================================================
# 5. 教师模型
# CNN 与 GCN 独立，最后融合
# ============================================================

class EEGTeacher(nn.Module):

    def __init__(
        self,
        elec_names,
        embed_dim=32,
        time_steps=80,
        hidden_size=10,
        dropout=0.2
    ):
        super().__init__()

        self.elec_names = elec_names
        self.time_steps = time_steps

        # -------------------------
        # CNN 分支
        # -------------------------
        self.common_conv = CommonConv2d()

        # -------------------------
        # GCN 分支
        # -------------------------
        self.gcn_branch = GCNBranch(
            elec_names=elec_names,
            input_dim=2560 // time_steps,
            hidden_dim=embed_dim,
            time_steps=time_steps
        )

        # -------------------------
        # 特征融合
        # CNN: 32维
        # GCN: 32维
        # 融合后: 64 -> 32
        # -------------------------
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # 电极维度的可学习聚合权重
        self.electrode_score = nn.Linear(embed_dim, 1)

        # -------------------------
        # 原有 BiLSTM 分类器
        # -------------------------
        self.Bilstm = BILSTMClassify(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=1,
            dropout=dropout,
            temporal_attention=False,
            channal_attention=False
        )

    def forward(self, x, return_features=False):
        # x: (B, 6, 2560)
        B, N, L = x.shape

        # =====================================
        # CNN 分支
        # =====================================

        # (B, 6, L) -> (B, 1, 6, L)
        cnn_x = x.unsqueeze(1)

        # (B, 32, 6, 80)
        cnn_x = self.common_conv(cnn_x)

        # (B, 32, 6, 80) -> (B, 80, 6, 32)
        cnn_features = cnn_x.permute(
            0, 3, 2, 1
        ).contiguous()

        # =====================================
        # GCN 分支
        # =====================================

        # (B, 80, 6, 32)
        gcn_features = self.gcn_branch(x)

        # =====================================
        # 特征融合
        # =====================================

        # (B, 80, 6, 64)
        fused = torch.cat(
            [cnn_features, gcn_features],
            dim=-1
        )

        # (B, 80, 6, 32)
        fused = self.fusion(fused)

        # =====================================
        # 电极维度聚合
        # =====================================

        # (B, 80, 6, 1)
        electrode_weights = F.softmax(
            self.electrode_score(fused),
            dim=2
        )

        # (B, 80, 32)
        temporal_features = torch.sum(
            fused * electrode_weights,
            dim=2
        )

        # =====================================
        # BiLSTM + 分类
        # =====================================

        # (B, 1)
        logits = self.Bilstm(temporal_features)

        if return_features:
            return {
                "logits": logits,
                "cnn_features": cnn_features,
                "gcn_features": gcn_features,
                "fused_features": fused,
                "temporal_features": temporal_features,
                "electrode_weights": electrode_weights
            }

        return logits